In [1]:
import os, sys, json, types
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)
print(f'Project root added to sys.path: {project_root}')

N_CORES = 2
os.environ['OMP_NUM_THREADS']      = str(N_CORES)
os.environ['MKL_NUM_THREADS']      = str(N_CORES)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_CORES)
os.environ['NUMEXPR_NUM_THREADS']  = str(N_CORES)

import numpy as np
import pybamm
import srcs
from filterpy.kalman import JulierSigmaPoints
from filterpy.kalman import UnscentedKalmanFilter as UKF
import scipy
from scipy.optimize import curve_fit
from tqdm import tqdm
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    'font.family'         : 'serif',
    'font.size'           : 10,
    'axes.labelsize'      : 10,
    'xtick.labelsize'     : 9,
    'ytick.labelsize'     : 9,
    'legend.fontsize'     : 9,
    'axes.prop_cycle'     : mpl.cycler('color', [
        '#0072B2', '#D55E00', '#009E73',
        '#E69F00', '#CC79A7', '#56B4E9',
    ]),
    'lines.linewidth'     : 1.5,
    'axes.linewidth'      : 0.8,
    'xtick.direction'     : 'in',
    'ytick.direction'     : 'in',
    'xtick.minor.visible' : True,
    'ytick.minor.visible' : True,
    'xtick.major.size'    : 4,
    'ytick.major.size'    : 4,
    'xtick.minor.size'    : 2,
    'ytick.minor.size'    : 2,
    'xtick.major.width'   : 0.8,
    'ytick.major.width'   : 0.8,
    'xtick.minor.width'   : 0.6,
    'ytick.minor.width'   : 0.6,
    'lines.markersize'    : 4,
    'errorbar.capsize'    : 3,
    'axes.xmargin'        : 0.02,
    'axes.ymargin'        : 0.02,
    'legend.frameon'      : False,
    'savefig.bbox'        : 'tight',
    'savefig.dpi'         : 300,
    # LaTeX text backend deliberately DISABLED: the local MiKTeX install errors on
    # invocation ("you have not checked for MiKTeX updates"), which propagates up
    # through tight_layout()/savefig() and crashes the run — even though `latex` is
    # on PATH, so a which('latex') check wrongly enables it. matplotlib's built-in
    # mathtext (Computer Modern) renders every label with no external TeX dependency.
    'text.usetex'         : False,
    'mathtext.fontset'    : 'cm',
})

import srcs.utils as utils
import srcs.utils_plotting as uplt

pybamm.set_logging_level('ERROR')
np.random.seed(42)

font_lab = {'family': 'serif', 'color': 'black', 'size': 12}
font_tit = {'family': 'serif', 'color': 'black', 'size': 14}

Project root added to sys.path: c:\Users\alexa\OneDrive - Politecnico di Milano\Desktop\Batteries\Python codes\SOC SOH Coestimation with PIMIONet\State estimation


In [2]:
# Papermill parameters — only plain assignments here
pm         = 'Mohtat2020'   # PyBaMM parameter set
experiment = 'exp2'      # subfolder under Data/Simulate_cycles/<pm>/


In [3]:
from pathlib import Path


# ════════════════════════════════════════════════════════════════════════════
# 1. PLANT SELECTION   (everything below is derived from model_config.json)
# ════════════════════════════════════════════════════════════════════════════
PROJECT_ROOT = srcs.utils.get_project_root()
print(f"Project root directory: {PROJECT_ROOT}")

# ── Chemistry selection — change pm to switch parameter sets and lookup tables ──
# Plant dataset to monitor — pick the folder under Data/Simulate_cycles/<pm>/<exp>.
folder_path = PROJECT_ROOT / 'State estimation' / 'Data' / 'Simulate_cycles' / pm / experiment
plant_dir   = folder_path / 'Plant'

with open(plant_dir / 'model_config.json', 'r') as f:
    plant_info = json.load(f)

# Chemistry-dependent fields read straight from the plant metadata.
N_cycles = float(plant_info['Ncycles'])
T_amb    = float(plant_info['temperature'])

param = pybamm.ParameterValues(pm)


# ════════════════════════════════════════════════════════════════════════════
# 2. SOH REFERENCE SOURCE  — always 'capacity' for ECM-based estimation
# ════════════════════════════════════════════════════════════════════════════
SOH_source = 'capacity'

Initial_capacity = float(plant_info.get('Initial capacity [A.h]',
                                        plant_info.get('Nominal capacity Ah', 1.0)))
print(f"SOH ref. source: '{SOH_source}' | Initial capacity = {Initial_capacity:.4f} Ah")
C_nom_Ah = Initial_capacity
print(f"Plant : pm='{pm}' | T_amb={T_amb:.1f}°C | Ncycles={N_cycles:.0f} | C_nom={C_nom_Ah:.3f} Ah")
# ════════════════════════════════════════════════════════════════════════════
# 3. RESULTS FOLDER   (auto-increment, never overwrite)
# ════════════════════════════════════════════════════════════════════════════
results_dir = folder_path / 'ECMD results'
results_dir.mkdir(parents=True, exist_ok=True)
i = 1
while True:
    candidate = results_dir / str(i)
    if not candidate.exists():
        candidate.mkdir()
        break
    i += 1
results_dir = candidate
print(f'Results will be saved to: {results_dir}')


Project root directory: C:\Users\alexa\OneDrive - Politecnico di Milano\Desktop\Batteries\Python codes\SOC SOH Coestimation with PIMIONet
SOH ref. source: 'capacity' | Initial capacity = 4.9676 Ah
Plant : pm='Mohtat2020' | T_amb=25.0°C | Ncycles=109 | C_nom=4.968 Ah
Results will be saved to: C:\Users\alexa\OneDrive - Politecnico di Milano\Desktop\Batteries\Python codes\SOC SOH Coestimation with PIMIONet\State estimation\Data\Simulate_cycles\Mohtat2020\exp2\ECMD results\8


## Naming conventions
**UPPER CASE** → plant variables: `SOC`, `LLI`, `Voltage_plant`, `Current`  
**lower case** → estimated quantities: `soc`, `soc_std`, `soh`, `soh_std`, `A_deg_est`

## Dual ECM filter architecture
- **Inner UKF**: states `[SOC, V1, V2, V3]`, dim_x=4. Measurement: terminal voltage.
- **Outer UKF**: states `[SOH, A_deg]`, dim_x=2. **Both** the capacity fade and the
  rate constant are estimated jointly online.

### Outer state model
$$\mathrm{SOH}_{k+1} = \mathrm{SOH}_k \cdot \exp\!\bigl(-A_k \cdot \Delta\mathrm{EFC}\bigr)$$
$$A_{k+1} = A_k \quad \text{(random walk — learned from data)}$$

Measurement: $y = C_\mathrm{est}/C_\mathrm{nom}$ from charge throughput / SOC swing.  
Trigger: cumulative $|\Delta\mathrm{SOC}| \geq \mathrm{EFC\_threshold}$.

In [4]:
# ── Load chemistry-specific KF parameters ─────────────────────────────────
# All ECM tuning values live in State estimation/configs/kf_params.json
# under the "ecm" sub-key — edit there, not here.
kfp     = utils.load_kf_params(pm)
ecm_cfg = kfp['ecm']

# ── Temperature-aware mode ────────────────────────────────────────────────────
ECM_MAT_PATH = str(PROJECT_ROOT / 'State estimation' / 'Data' / 'Parameters'
                    / f'lookuptables_ECM_{pm}.mat')
ecm = srcs.ECM.ecm_T(ECM_MAT_PATH, C=C_nom_Ah)
print('ECM model : ecm_T  (2-D Temperature × SOC lookup tables)')

ecm.A_deg_init = ecm_cfg['outer_ukf']['A_deg_init']
ecm.sigma_I    = kfp['NOISE_STD']

# 1. Inner state function ─────────────────────────────────────────────────────
def _FD_inner(self, x, dt, I, soh=1.0, T=298.15):
    x_aug = np.concatenate([x[:4], np.array([soh])])
    return self.F(x_aug, dt, I, T)[:4]

# 2. Inner measurement function ───────────────────────────────────────────────
def _HD_inner(self, x, I, soh=1.0, T=298.15):
    x_aug = np.concatenate([x[:4], np.array([soh])])
    return self.H(x_aug, I, T)

# 3. Outer state function — joint SOH + A estimation ─────────────────────────
def _FD1_soh(self, x, dt=None, dEFC=1):
    soh_next = x[0] * np.exp(-x[1] * dEFC)
    A_next   = x[1]
    return np.array([
        np.clip(soh_next, 0.50, 1.00),
        np.clip(A_next,   1e-6, 0.10),
    ])

# 4. Outer measurement function ───────────────────────────────────────────────
def _HD1_soh(self, x):
    return np.array([x[0]])

ecm.FD_inner = types.MethodType(_FD_inner, ecm)
ecm.HD_inner = types.MethodType(_HD_inner, ecm)
ecm.FD1_soh  = types.MethodType(_FD1_soh,  ecm)
ecm.HD1_soh  = types.MethodType(_HD1_soh,  ecm)

print(f'Outer state : [SOH, A_deg]  — A_deg estimated online')
print(f'A_deg init  : {ecm.A_deg_init:.3e} [1/EFC]')

ECM initialized with C=4.968 Ah
ECM model : ecm_T  (2-D Temperature × SOC lookup tables)
Outer state : [SOH, A_deg]  — A_deg estimated online
A_deg init  : 3.000e-03 [1/EFC]


In [5]:
noise_std        = kfp['NOISE_STD']
T_w              = ecm_cfg['T_W']
ukf_update_every = ecm_cfg['ukf_update_every']
ik_ecm           = ecm_cfg['inner_ukf']
ok_ecm           = ecm_cfg['outer_ukf']

# Inner UKF: [SOC, V1, V2, V3]  dim_x=4, dim_z=1
sigmas = JulierSigmaPoints(n=4, kappa=2, sqrt_method=scipy.linalg.sqrtm)
ukf    = UKF(dim_x=4, dim_z=1, dt=T_w, fx=ecm.FD_inner, hx=ecm.HD_inner,
              points=sigmas)

ukf.x  = np.array([1, 0.0, 0.0, 0.0])
ukf.P  = np.diag(ik_ecm['P0_diag'])
ukf.R *= ik_ecm['R']
ukf.Q  = np.diag([(c * T_w)**2 for c in ik_ecm['Q_coeff']])

# ── Outer UKF: [SOH, A_deg]  dim_x=2, dim_z=1 ────────────────────────────────
sigmas1 = JulierSigmaPoints(n=2, kappa=0, sqrt_method=scipy.linalg.sqrtm)
ukf1    = UKF(dim_x=2, dim_z=1, dt=1,
              fx=ecm.FD1_soh, hx=ecm.HD1_soh,
              points=sigmas1)

ukf1.x  = np.array([1.00, ecm.A_deg_init])
ukf1.P  = np.diag(ok_ecm['P0_diag'])
ukf1.Q  = np.diag(ok_ecm['Q_diag'])
ukf1.R  = np.eye(1) * ok_ecm['R_base']
R_base  = ok_ecm['R_base']
R_SF    = ok_ecm['R_SF']

obs_info = {
    'noise_std'    : noise_std, 'T_w': T_w, 'ukf_update_every': ukf_update_every,
    'inner_states' : ['SOC', 'V1', 'V2', 'V3'],
    'outer_states' : ['SOH fraction [0.5, 1.0]', 'A_deg [1/EFC]'],
    'outer_model'  : 'SOH(EFC) = exp(-A_deg * EFC)  — A_deg estimated online',
    'A_deg_init'   : ecm.A_deg_init,
    'kappa_inner'  : sigmas.kappa, 'kappa_outer': sigmas1.kappa,
}
with open(results_dir / 'obs_config.json', 'w') as f:
    json.dump(obs_info, f, indent=4)

print(f'Outer UKF state : [SOH, A_deg]  (dim_x=2)')
print(f'  x0 = {ukf1.x}')
print(f'  P0 = diag({np.diag(ukf1.P)})')

Outer UKF state : [SOH, A_deg]  (dim_x=2)
  x0 = [1.    0.003]
  P0 = diag([1.e-06 1.e-05])


## Main estimation loop — Dual ECM UKF with EFC-based outer trigger

Identical structure to the Dual EM filter (v5) for a fair comparison:
same `BatteryTracker`, same EFC logic, same plotting calls, same CSV export.

**Outer measurement**: normalised capacity `C_est/C_nom` estimated from
Ah throughput and the SOC swing over the EFC window.
This is the physically meaningful outer observable for an ECM observer
(no electrochemical states available for Li conservation).

In [6]:
EFC_threshold = ecm_cfg['EFC_THRESHOLD']   # [%] cumulative |ΔSOC| to trigger outer update

# -- Rest-anchored capacity (robust to dynamic/DST currents) -----------------
# Cut the SOC segment at QUALIFIED rests (|I|~0 for > t_rest_min): SOC is read only
# where it is OCV-reliable; charge between two rests is a model-free integral.
USE_REST_ANCHOR = True
i_rest     = 0.1     # |I| below this = rest [A] (~0.02C for a 5 Ah cell)
t_rest_min = 300.0   # a rest must exceed this to qualify as an anchor [s]
dsoc_min   = 40.0    # minimum |dSOC| between anchors to accept a capacity obs [%]
cycle   = 1
D_cycle = 1
P_soc_old = 1

tracker = utils.BatteryTracker(filter_type='dual')
summary_per_cycle = []   # standard artifact for make_paper_figures.py
detailed_cycles   = []

A_log_efc = []   # EFC at each outer update
A_log_val = []   # estimated A_deg
A_log_std = []   # 1-sigma uncertainty of A_deg

est_soh_pct = 200.0   # will be updated at first outer trigger
est_soh_std = 0.0

while True:

    out           = utils.load_cycle(folder_path=plant_dir, k=cycle, plot=False)
    Time          = out['time']
    Voltage_plant = out['voltage'] + np.random.normal(0, noise_std, len(out['voltage']))
    Current       = out['current'] + np.random.normal(0, noise_std, len(out['current']))
    SOC           = out['soc']
    LLI           = out['lli']
    Temp          = out['Temp [C]']
    Capacity_cycle = out.get('capacity', None)

    if Capacity_cycle is not None:
        current_soh_ref = 100.0 * Capacity_cycle / Initial_capacity
    else:
        current_soh_ref = 100.0 - LLI

    time_log = []; soc_log = []; soc_std_log = []; soc_err_log = []; V_log = []
    tt = 0.0;  I_wind = []
    predict_count = 0   # reset each cycle; update fires every ukf_update_every steps

    soh_est_frac = 1.0

    for j in tqdm(range(len(Time)), desc=f'Cycle {cycle:3d} | inner UKF (ECM)'):

        dT = Time[j+1] - Time[j] if j < len(Time) - 1 else 2.0
        tt += dT
        I_wind.append(Current[j])

        if tt >= T_w or (j > 1 and abs(Current[j] - Current[j-1]) > 0.5):

            I_avg = float(np.mean(I_wind))
            T_K   = float(Temp[j]) + 273.15 if float(Temp[j]) < 200 else float(Temp[j])

            predict_count += 1
            ukf.predict(I=I_avg, dt=tt, soh=soh_est_frac, T=T_K)

            if predict_count % ukf_update_every == 0:
                z_inner = np.array([Voltage_plant[j]])
                ukf.update(z_inner, I=Current[j], soh=soh_est_frac, T=T_K)

            tracker.curr.append(I_avg)
            tracker.dt.append(tt)

            soc_avg = np.clip(ukf.x[0] * 100.0, 0.0, 100.0)
            soc_std = np.sqrt(ukf.P[0, 0]) * 100.0

            time_log.append(Time[j])
            soc_log.append(soc_avg)
            soc_std_log.append(soc_std)
            soc_err_log.append(soc_avg - SOC[j])
            V_log.append(ecm.HD_inner(ukf.x, I=Current[j], soh=soh_est_frac, T=T_K)[0])

            tracker.update_efc(soc_avg)
            tracker.update_swing(soc_avg, I_avg)
            cap_obs_rest = tracker.rest_anchor(soc_avg, I_avg, tt,
                                               i_rest=i_rest, t_rest_min=t_rest_min, dsoc_min=dsoc_min)

            # ── Outer filter trigger ──────────────────────────────────────────
            _outer_trigger = (cap_obs_rest is not None) if USE_REST_ANCHOR else (tracker.efc_accum >= EFC_threshold)
            if _outer_trigger and len(tracker.curr) > 10:

                curr_arr   = np.array(tracker.curr)
                dt_arr     = np.array(tracker.dt)

                delta_Q_Ah       = float(np.sum(np.abs(curr_arr) * dt_arr) / 3600.0)
                delta_soc_window = tracker.efc_accum / 100.0
                soc_swing = tracker.swing_total(soc_avg)
                C_est_Ah         = delta_Q_Ah / (soc_swing / 100.0)       # segmented estimate (fallback); swing is in %
                # Rest-anchored SOH: capacity read only at OCV-reliable rests (robust to dynamic/DST currents)
                soh_meas         = (cap_obs_rest / C_nom_Ah) if USE_REST_ANCHOR else (C_est_Ah / C_nom_Ah)

                ukf1.predict(soc_swing)

                P_soc_new        = (soc_std_log[-1] / 100)**2
                R_outer_adaptive = (P_soc_new + P_soc_old) / 2
                P_soc_old        = P_soc_new
                ukf1.R           = np.eye(1) * (R_base + R_SF * R_outer_adaptive)

                ukf1.update(z=np.array([soh_meas]))

                soh_est_frac = float(ukf1.x[0])

                est_soh_pct = soh_est_frac * 100.0
                est_soh_std = np.sqrt(ukf1.P[0, 0]) * 100.0
                est_A_deg   = float(ukf1.x[1])
                est_A_std   = float(np.sqrt(ukf1.P[1, 1]))

                A_log_efc.append(tracker.total_efc)
                A_log_val.append(est_A_deg)
                A_log_std.append(est_A_std)

                tracker.log_global_soh(
                    current_soh_raw     = est_soh_pct,
                    current_soh_std_raw = est_soh_std,
                    current_soh_ref     = current_soh_ref,
                    cycle               = cycle,
                )

                print(f'\n  -- Outer UKF update (EFC={tracker.total_efc:.2f}, T={T_K-273.15:.1f}°C) --')
                print(f'     GT SOH     : {current_soh_ref:.2f} % -- Estimated {est_soh_pct:.2f} ± {est_soh_std:.2f} % (1σ)')
                print(f'     A_deg est  : {est_A_deg:.4e} ± {est_A_std:.1e}  [1/EFC]')

                tracker.reset_buffer()

            tt = 0.0;  I_wind = []

    # ── End-of-cycle bookkeeping ──────────────────────────────────────────────
    soc_err_arr = np.array(soc_err_log)
    soc_std_arr = np.array(soc_std_log)
    soc_rmse    = float(np.sqrt(np.mean(soc_err_arr**2)))
    soc_cwc     = utils.compute_cwc(np.zeros_like(soc_err_arr),
                                     soc_err_arr - 2*soc_std_arr,
                                     soc_err_arr + 2*soc_std_arr)
    print(f'RMSE SOC / CWC : {soc_rmse:.2f} % / {soc_cwc:.4f}')

    tracker.cycles.append(cycle)
    tracker.soc_rmse_cycles.append(soc_rmse)
    tracker.soc_cwc_cycles.append(soc_cwc)
    tracker.cycle_end_efc.append(tracker.total_efc)
    summary_per_cycle.append(dict(cycle=cycle, efc=tracker.total_efc,
        soc_rmse=soc_rmse, soc_cwc=soc_cwc, soh_avg=est_soh_pct,
        soh_end=est_soh_pct, soh_std=est_soh_std, soh_ref=current_soh_ref))
    if cycle in (1, max(1, int(N_cycles)//2), int(N_cycles)):
        detailed_cycles.append(dict(cycle=cycle, Time_plant=Time, SOC_plant=SOC,
            time_log=time_log, soc_log=soc_log, soc_std_log=soc_std_log))

    tracker.save_cycle_data(
        cycle          = cycle,
        time_log       = time_log,
        v_all          = V_log,
        i_all          = Current,
        soc_log        = soc_log,
        soc_std_log    = soc_std_log,
        soc_err_log    = soc_err_log,
        current_soh    = est_soh_pct,
        current_soh_std= est_soh_std,
        results_dir    = results_dir,
    )

    # -- Per-cycle diagnostics: estimated quantities --------------------------
    if tracker.hist_soh_est:                      # skip until first outer update
        tracker.plot_soh_history()

    uplt.plot_cycle_diagnostics(
        cycle, time_log, Time,
        soc_log, soc_std_log, soc_err_log, SOC,
        save=False, show=True
    )

    # Voltage tracking: measured plant voltage vs ECM-predicted voltage
    fig_v, ax_v = plt.subplots(figsize=(9, 3.2))
    ax_v.plot(np.asarray(Time) / 60.0, Voltage_plant, lw=0.7, alpha=0.55,
              color='tab:blue', label='plant (meas)')
    ax_v.plot(np.asarray(time_log) / 60.0, V_log, lw=1.1,
              color='tab:orange', label='ECM predicted')
    ax_v.set_xlabel('Time [min]'); ax_v.set_ylabel('Voltage [V]')
    ax_v.set_title(f'Cycle {cycle} - voltage tracking')
    ax_v.grid(True, ls=':', alpha=0.6); ax_v.legend(fontsize=9)
    plt.show(); plt.close(fig_v)


    cycle += D_cycle
    if cycle > N_cycles:
        break

Cycle   1 | inner UKF (ECM):   0%|          | 16/8802 [00:00<00:57, 152.65it/s]

Cycle   1 | inner UKF (ECM):  54%|█████▎    | 4715/8802 [00:32<00:28, 145.63it/s]


  -- Outer UKF update (EFC=0.49, T=25.0°C) --
     GT SOH     : 99.87 % -- Estimated 99.66 ± 0.53 % (1σ)
     A_deg est  : 3.3682e-03 ± 2.7e-03  [1/EFC]


Cycle   1 | inner UKF (ECM): 100%|██████████| 8802/8802 [00:59<00:00, 146.94it/s]


RMSE SOC / CWC : 1.10 % / 14.4176


...

RuntimeError: latex was not able to process the following string:
b'\\xb12\\u03c3 Confidence'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.28 (MiKTeX 25.12) (preloaded format=latex.fmt)
 restricted \write18 enabled.
entering extended mode
(file.tex
LaTeX2e <2025-11-01>
L3 programming layer <2025-12-24>
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/base\article.cls
Document Class: article 2025/01/22 v1.4n Standard LaTeX document class
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/base\size10.clo))
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/type1cm\type1cm.sty)
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/cm-super\type1ec.sty
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/base\t1cmr.fd))
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/base\inputenc.sty)
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/lat

Error in callback <function _draw_all_if_interactive at 0x000001F40C498680> (for post_execute), with arguments args (),kwargs {}:


RuntimeError: latex was not able to process the following string:
b'\\xb12\\u03c3 Confidence'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.28 (MiKTeX 25.12) (preloaded format=latex.fmt)
 restricted \write18 enabled.
entering extended mode
(file.tex
LaTeX2e <2025-11-01>
L3 programming layer <2025-12-24>
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/base\article.cls
Document Class: article 2025/01/22 v1.4n Standard LaTeX document class
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/base\size10.clo))
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/type1cm\type1cm.sty)
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/cm-super\type1ec.sty
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/base\t1cmr.fd))
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/base\inputenc.sty)
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/lat

RuntimeError: latex was not able to process the following string:
b'\\xb12\\u03c3 Confidence'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.28 (MiKTeX 25.12) (preloaded format=latex.fmt)
 restricted \write18 enabled.
entering extended mode
(file.tex
LaTeX2e <2025-11-01>
L3 programming layer <2025-12-24>
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/base\article.cls
Document Class: article 2025/01/22 v1.4n Standard LaTeX document class
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/base\size10.clo))
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/type1cm\type1cm.sty)
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/cm-super\type1ec.sty
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/base\t1cmr.fd))
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/latex/base\inputenc.sty)
(C:\Users\alexa\AppData\Local\Programs\MiKTeX\tex/lat

<Figure size 900x500 with 1 Axes>

## Summary plots across all cycles
SOH and SOC metrics vs Equivalent Full Cycles.

## True A — fit exponential model to reference SOH

Fit $\mathrm{SOH}(\mathrm{EFC}) = \exp(-A_\mathrm{true} \cdot \mathrm{EFC})$ to the ground-truth reference SOH data collected by the tracker during the run. This gives the "oracle" value against which the online estimate $\hat{A}$ is benchmarked.

In [ ]:
def _exp_model(efc, A):
    return np.exp(-A * efc)

# Reference SOH (fraction) and EFC collected by the tracker during the run
efc_ref_arr = np.array(tracker.hist_efc)
soh_ref_arr = np.array(tracker.hist_soh_ref) / 100.0   # convert % → fraction

# Non-linear least squares: SOH_frac = exp(-A * EFC)
popt, pcov   = curve_fit(_exp_model, efc_ref_arr, soh_ref_arr,
                          p0=[3e-3], bounds=(0, np.inf))
A_true       = float(popt[0])
A_true_std   = float(np.sqrt(pcov[0, 0]))   # 1-sigma from covariance

# Cross-check: log-space linear fit  ln(SOH) = -A * EFC
log_soh_ref  = np.log(np.clip(soh_ref_arr, 1e-9, None))
A_true_log   = float(-np.polyfit(efc_ref_arr, log_soh_ref, 1)[0])

# Goodness of fit
soh_pred_ref = _exp_model(efc_ref_arr, A_true)
rmse_pct     = float(np.sqrt(np.mean((soh_pred_ref - soh_ref_arr)**2))) * 100.0

print('=' * 55)
print('  TRUE A — exponential fit to reference SOH')
print('=' * 55)
print(f'  Non-linear fit : A_true = {A_true:.4e}  ±  {A_true_std:.1e}  [1/EFC]')
print(f'  Log-space check: A_true = {A_true_log:.4e}  [1/EFC]')
print(f'  Fit RMSE       : {rmse_pct:.4f} % SOH')
if A_log_val:
    print()
    print(f'  Final online A : {A_log_val[-1]:.4e}  ±  {A_log_std[-1]:.1e}  [1/EFC]')
    print(f'  Relative error : {abs(A_log_val[-1] - A_true) / A_true * 100:.2f} %')
print('=' * 55)

  TRUE A — exponential fit to reference SOH
  Non-linear fit : A_true = 3.0497e-03  ±  4.5e-06  [1/EFC]
  Log-space check: A_true = 2.9950e-03  [1/EFC]
  Fit RMSE       : 0.1964 % SOH

  Final online A : 9.9992e-02  ±  5.5e-06  [1/EFC]
  Relative error : 3178.71 %


In [ ]:
print('\nGenerating final experiment summaries...')

# ── SOH estimation metrics ────────────────────────────────────────────────────
soh_est_arr = np.array(tracker.hist_soh_est)
soh_ref_arr = np.array(tracker.hist_soh_ref)
soh_std_arr = np.array(tracker.hist_soh_std)
soh_err_arr = soh_est_arr - soh_ref_arr

soh_rmse = float(np.sqrt(np.mean(soh_err_arr ** 2)))
soh_cwc  = utils.compute_cwc(
    np.zeros_like(soh_err_arr),
    soh_err_arr - 2 * soh_std_arr,
    soh_err_arr + 2 * soh_std_arr,
)
print(f'SOH  RMSE : {soh_rmse:.4f} %')
print(f'SOH  CWC  : {soh_cwc:.4f}')

# ── 1. SOH vs EFC ────────────────────────────────────────────────────────────
plt.figure(figsize=(9, 5))
plt.plot(tracker.hist_efc, soh_est_arr, 'b.-', label='Estimated SOH [%]', markersize=6)
plt.plot(tracker.hist_efc, soh_ref_arr, 'k--', label='Reference SOH [%]', linewidth=2)
plt.fill_between(tracker.hist_efc, soh_est_arr - 2*soh_std_arr, soh_est_arr + 2*soh_std_arr,
                  color='blue', alpha=0.2, label='$\\pm 2\\sigma$')
plt.xlabel('Equivalent Full Cycles (EFC)')
plt.ylabel('State of Health [%]')
plt.title(f'Dual ECM Filter — SOH Estimation  (RMSE={soh_rmse:.3f} %, CWC={soh_cwc:.3f})')
plt.legend(); plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.savefig(results_dir / 'SOH_estimation_vs_EFC.png', dpi=300)
plt.show()

# ── 2. A_deg convergence ─────────────────────────────────────────────────────
if A_log_efc:
    A_arr     = np.array(A_log_val)
    A_std_arr = np.array(A_log_std)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.fill_between(A_log_efc, A_arr - 2*A_std_arr, A_arr + 2*A_std_arr,
                    color='#009E73', alpha=0.2, label=r'$\pm 2\sigma$')
    ax.plot(A_log_efc, A_arr, 'o-', color='#009E73', markersize=5,
            label=r'Estimated $\hat{A}$')
    ax.axhline(A_true, color='k', linewidth=2.0, linestyle='--',
               label=f'True $A$ = {A_true:.4e}  [1/EFC]')
    ax.axhspan(A_true - A_true_std, A_true + A_true_std,
               color='gray', alpha=0.15, label=r'True $A$ fit $\pm1\sigma$')
    ax.set_xlabel('Equivalent Full Cycles (EFC)')
    ax.set_ylabel('$A$ [1/EFC]')
    ax.set_title('Online identification of exponential fade-rate $A$')
    ax.legend(); ax.grid(True, linestyle=':', alpha=0.7)
    plt.tight_layout()
    plt.savefig(results_dir / 'A_deg_convergence.png', dpi=300)
    plt.show()

    print(f'\nTrue   A : {A_true:.4e}  ±  {A_true_std:.1e}  [1/EFC]')
    print(f'Final  A : {A_arr[-1]:.4e}  ±  {A_std_arr[-1]:.1e}  [1/EFC]')
    print(f'Rel. err : {abs(A_arr[-1] - A_true) / A_true * 100:.2f} %')
    print(f'  (use Final A as ecm.A_deg_init in future runs)')

# ── 3. SOC RMSE & CWC ────────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(9, 5))
l1, = ax1.plot(tracker.cycle_end_efc, tracker.soc_rmse_cycles, 'o-', label='RMSE SOC [%]', color='tab:red')
ax1.set_xlabel('Equivalent Full Cycles (EFC)')
ax1.set_ylabel('SOC error [%]', color=l1.get_color())
ax1.tick_params(axis='y', colors=l1.get_color())
ax1.spines['left'].set_color(l1.get_color())
ax1.set_title('Dual ECM Filter - SOC Estimation Error over Battery Life')
ax1.grid(True, linestyle=':', alpha=0.7)
ax2 = ax1.twinx()
l2, = ax2.plot(tracker.cycle_end_efc, tracker.soc_cwc_cycles, 's--', label='CWC', color='tab:blue')
ax2.set_ylabel('CWC', color=l2.get_color())
ax2.tick_params(axis='y', colors=l2.get_color())
ax2.spines['right'].set_color(l2.get_color())
ax1.legend(handles=[l1, l2], loc='upper center')
plt.tight_layout()
plt.savefig(results_dir / 'SOC_estimation_rmse_cwc_vs_EFC.png', dpi=300)
plt.show()

# ── 4. Save CSVs ─────────────────────────────────────────────────────────────
df_soh = pd.DataFrame({
    'Cumulative EFC'   : tracker.hist_efc,
    'Estimated SOH [%]': tracker.hist_soh_est,
    'Reference SOH [%]': tracker.hist_soh_ref,
    'SOH STD [%]'      : tracker.hist_soh_std,
    'SOH error [%]'    : soh_err_arr,
})
df_soh.to_csv(results_dir / 'Summary_SOH_History.csv', index=False)

df_metrics = pd.DataFrame({
    'Cycle'           : tracker.cycles,
    'End-of-Cycle EFC': tracker.cycle_end_efc,
    'SOC RMSE [%]'    : tracker.soc_rmse_cycles,
    'CWC'             : tracker.soc_cwc_cycles,
})
df_metrics.to_csv(results_dir / 'Summary_SOC_Metrics.csv', index=False)

if A_log_efc:
    df_A = pd.DataFrame({
        'Cumulative EFC' : A_log_efc,
        'A_deg [1/EFC]'  : A_log_val,
        'A_deg STD'      : A_log_std,
    })
    df_A.to_csv(results_dir / 'Summary_A_deg_convergence.csv', index=False)

# ── 5. Save scalar SOH metrics ────────────────────────────────────────────────
soh_metrics = {'SOH RMSE [%]': soh_rmse, 'SOH CWC': soh_cwc}
with open(results_dir / 'SOH_metrics.json', 'w') as f:
    json.dump(soh_metrics, f, indent=4)

print(f'\nExperiment complete! Results saved to: {results_dir}')

# ── Standard artifacts consumed by make_paper_figures.py ──
utils.save_run_artifacts(results_dir, summary_per_cycle, detailed_cycles,
    meta=dict(method='ECMD results', estimator='ECM-UKF', pm=pm, experiment=experiment))
print('Standard artifacts saved to', results_dir)



Generating final experiment summaries...
SOH  RMSE : 36.7872 %
SOH  CWC  : 201.0229
